# Shrinking a shock-advice network to fit an AED chip

ESP3201A final project (NUS, 2026). An automated external defibrillator (AED) looks at 5 seconds of ECG and decides **"shock" or "no shock"**. That decision runs on a small chip, so the model has to be tiny.

This notebook runs the whole project from start to finish:

1. **Download** three public ECG databases from PhysioNet and cut them into 5-second clips.
2. **Train** a large 1D ResNet and a small CNN.
3. **Compress** the large model: structured pruning (remove whole filters), unstructured pruning (zero single weights) and int8 quantisation (store numbers in 8 bits instead of 32).
4. **Score** every model against the American Heart Association (AHA) targets for each rhythm, and check whether it fits an **STM32L432 chip (256 KiB flash, 64 KiB RAM)**.
5. **Draw** the report's figures and run a **demo** that prints "SHOCK ADVISED" or "NO SHOCK".

**How to run:** `Runtime → Run all`. A GPU runtime (`Runtime → Change runtime type → T4 GPU`) makes training faster, but CPU also works.

The code is the same as the `code/` folder of the GitHub repository; each script is written to disk by one cell below, so you can read it here.

## 0. Setup

In [ ]:
#@title Install the one extra library and check the runtime
!pip -q install wfdb
!rm -rf code && mkdir -p code
import tensorflow as tf, os
print("TensorFlow", tf.__version__, "| CPUs:", os.cpu_count(), "| GPU:", tf.config.list_physical_devices("GPU") or "none")

## 1. The code

Each cell below writes one script into `code/`. Run them to create the files; open them to read the code.

| Script | What it does |
|---|---|
| `config.py` | All settings: paths, clip format, training budget, chip budget, AHA targets |
| `data.py` | Downloads the ECG databases, cuts 5 s clips at 125 Hz, labels each clip with its AHA rhythm category |
| `models.py` | The small CNN (4,625 weights) and the large 1D ResNet (501,537 weights) |
| `compress.py` | Structured and unstructured pruning, TFLite float32 / int8 conversion, memory and size measurement |
| `evaluate.py` | AHA scoring with a lower confidence limit from a bootstrap over patients |
| `run_experiments.py` | Trains, prunes, converts and scores every model on each patient-wise fold |
| `summarise.py` | Pools the folds into result tables and `RESULTS.md` |
| `make_figures.py` | Draws every figure in the report |
| `demo.py` | Runs the int8 model on ECG clips and prints the AED's answer |

In [ ]:
%%writefile code/config.py
"""Settings shared by every script: paths, signal format, training budget, target chip and the AHA targets."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"
RAW = DATA / "physionet"
CLIPS = DATA / "clips.npz"
RESULTS = ROOT / "results"

# Signal: one ECG lead, 5 s clips at 125 Hz = 625 samples per clip
FS, WIN_S = 125, 5
WIN = FS * WIN_S
DATABASES = ("cudb", "vfdb", "mitdb")

# Training
SEED, FOLDS = 0, 5
EPOCHS, FT_EPOCHS, BATCH = 40, 20, 128
N_CALIB = 300  # clips used to calibrate full-integer int8 (from training patients only)

# Compression settings
KEEPS = (0.75, 0.50, 0.35)    # structured pruning: fraction of filters kept in every layer
SCRATCH_KEEPS = (0.50, 0.35)  # same narrow shapes, trained from scratch (the "train small" control)
SPARSITY = 0.75               # unstructured pruning: fraction of weights set to zero in every layer

# Target chip: STM32 NUCLEO-L432KC (Arm Cortex-M4, 80 MHz)
FLASH_BYTES, RAM_BYTES, MCU_HZ = 256 * 1024, 64 * 1024, 80e6
MACS_PER_CYCLE = (0.5, 1.0)   # assumed int8 throughput range with CMSIS-NN kernels (estimate, not measured)

# AHA performance goals for AED rhythm analysis (Kerber et al., Circulation 1997;95:1677).
# (rhythm category, metric, goal, lowest acceptable one-sided 90% lower confidence limit)
AHA = [
    ("coarse VF", "sensitivity", 0.90, 0.87),
    ("rapid VT", "sensitivity", 0.75, 0.67),
    ("NSR", "specificity", 0.99, 0.97),
    ("other non-shockable", "specificity", 0.95, 0.88),
    ("asystole", "specificity", 0.95, 0.92),
]
REPORT_ONLY = ["fine VF", "other VT"]  # "intermediate" rhythms: the AHA asks for the result but sets no goal

# Rhythm labelling thresholds. VF amplitudes follow Kerber 1997. The AHA leaves the rapid-VT rate cut-off
# to the manufacturer; 180 bpm is our choice.
COARSE_VF_UV, ASYSTOLE_UV, RAPID_VT_BPM = 200, 100, 180

In [ ]:
%%writefile code/data.py
"""Step 1: download three PhysioNet databases and cut them into labelled 5 s single-lead clips.

    python code/data.py

Output: data/clips.npz holding X (clips x 625 samples, millivolts) and, for every clip, its database, record,
patient, start time, AHA rhythm category and shock label (1 shock, 0 no shock, -1 intermediate).
"""
import numpy as np, pandas as pd, wfdb
from scipy.signal import butter, filtfilt, find_peaks, resample_poly
from config import RAW, CLIPS, FS, WIN, DATABASES, COARSE_VF_UV, ASYSTOLE_UV, RAPID_VT_BPM

# Rhythm label in the annotation file -> rhythm group. None = drop (noise, paced or unclear rhythm).
RHYTHM = {"N": "NSR", "NSR": "NSR",
          "VF": "VF", "VFIB": "VF", "VFL": "VF", "VT": "VT", "ASYS": "ASYS",
          **{r: "OTHER" for r in ["AF", "AFIB", "AFL", "SVTA", "SBR", "B", "BI", "BII", "T", "NOD", "PREX", "IVR", "AB", "VER"]},
          **{r: None for r in ["NOISE", "P", "PM", "HGEA"]}}
SHOCK = {"coarse VF": 1, "rapid VT": 1, "NSR": 0, "other non-shockable": 0, "asystole": 0, "fine VF": -1, "other VT": -1}
SAME_PATIENT = {"mitdb/202": "mitdb/201"}  # MIT-BIH 201 and 202 come from the same person
LEAD = {"cudb": "ECG", "vfdb": "ECG", "mitdb": "MLII"}
_B, _A = butter(4, [0.5, 30], btype="band", fs=FS)


def download():
    """Fetch the databases from PhysioNet (about 130 MB) unless they are already in data/physionet."""
    for db in DATABASES:
        folder = RAW / db
        if not any(folder.glob("*.hea")):
            print(f"downloading {db} ...")
            wfdb.dl_database(db, str(folder))


def preprocess(x, fs):
    """Resample to 125 Hz and band-pass 0.5-30 Hz (removes baseline wander and muscle noise)."""
    up, down = (1, 2) if fs == 250 else (25, 72) if fs == 360 else (FS, int(fs))
    return filtfilt(_B, _A, resample_poly(np.nan_to_num(x), up, down))


def rhythm_track(db, rec, n):
    """Rhythm group of every sample, from the rhythm-change annotations (and CUDB's [ ] VF-episode markers).

    CUDB has rhythm notes in only some records, so whatever it leaves unmarked, including the stretch after a VF
    episode until the next note, stays None (dropped) instead of being assumed to be sinus rhythm.
    """
    a = wfdb.rdann(str(RAW / db / rec), "atr")
    track = np.full(n, None, dtype=object)
    changes = [(s, RHYTHM.get(x.strip("\x00")[1:], None)) for s, x in zip(a.sample, a.aux_note) if x.startswith("(")]
    for (s, r), (e, _) in zip(changes, changes[1:] + [(n, None)]):
        track[s:e] = r
    if db == "cudb":
        starts = [s for s, c in zip(a.sample, a.symbol) if c == "["]
        ends = [s for s, c in zip(a.sample, a.symbol) if c == "]"]
        for s in starts:
            e = next((x for x in ends if x > s), n)
            track[s:e] = "VF"
            nxt = next((c for c, _ in changes if c > e), n)
            track[e:nxt] = None
    return track


def amplitude_uv(x):
    """Median peak-to-peak amplitude over 1 s sub-windows, in microvolts."""
    return 1000 * np.median([np.ptp(x[i:i + FS]) for i in range(0, len(x), FS)])


def rate_bpm(x):
    """Heart rate from the autocorrelation: the earliest lag (40-300 bpm) whose peak reaches 70% of the strongest.

    Checked against MIT-BIH beat labels: median error 0.6 bpm, 99% agreement on the >= 180 bpm split.
    """
    x = x - x.mean()
    ac = np.correlate(x, x, "full")[len(x) - 1:]
    ac /= ac[0]
    pk, _ = find_peaks(ac[:int(1.5 * FS) + 1])
    pk = pk[(pk >= int(0.2 * FS)) & (ac[pk] > 0)]
    return 60 * FS / pk[np.argmax(ac[pk] >= 0.7 * ac[pk].max())] if len(pk) else 0.0


def aha_category(group, x):
    """Rhythm group -> AHA category. VF is split by amplitude, VT by rate."""
    if group == "VF":
        a = amplitude_uv(x)
        return "coarse VF" if a >= COARSE_VF_UV else "fine VF" if a >= ASYSTOLE_UV else None
    if group == "VT":
        return "rapid VT" if rate_bpm(x) >= RAPID_VT_BPM else "other VT"
    return {"NSR": "NSR", "OTHER": "other non-shockable", "ASYS": "asystole"}[group]


def build_clips():
    X, meta = [], []
    for db in DATABASES:
        for hea in sorted((RAW / db).glob("*.hea")):
            rec = hea.stem
            sig, f = wfdb.rdsamp(str(RAW / db / rec))
            if LEAD[db] not in f["sig_name"]:
                continue  # MIT-BIH 102 and 104 have no MLII lead (paced records)
            raw = sig[:, f["sig_name"].index(LEAD[db])]
            track = rhythm_track(db, rec, len(raw))
            x = preprocess(raw, f["fs"])
            track = track[(np.arange(len(x)) * f["fs"] / FS).astype(int)]
            pid = SAME_PATIENT.get(f"{db}/{rec}", f"{db}/{rec}")
            for s in range(0, len(x) - WIN + 1, WIN):  # non-overlapping 5 s clips
                g = set(track[s:s + WIN])
                if len(g) != 1 or None in g:
                    continue  # the rhythm changes inside the clip, or it is noise / unlabelled
                cat = aha_category(g.pop(), x[s:s + WIN])
                if cat is None:
                    continue
                X.append(x[s:s + WIN].astype(np.float32))
                meta.append(dict(db=db, record=rec, patient=pid, start_s=s / FS, category=cat, shock=SHOCK[cat]))
    meta = pd.DataFrame(meta)
    CLIPS.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(CLIPS, X=np.stack(X), **{c: meta[c].values for c in meta})

    counts = meta.pivot_table(index="category", columns="db", values="start_s", aggfunc="count", fill_value=0)
    counts["clips"] = counts.sum(axis=1)
    counts["patients"] = meta.groupby("category")["patient"].nunique()
    print(counts.reindex(list(SHOCK)).fillna(0).astype(int).to_string())
    print(f"\n{len(meta)} clips from {meta.patient.nunique()} patients -> {CLIPS}")


def load_clips():
    """Clips as a dict of arrays; X is clipped to +-5 mV and given a channel axis, ready for Conv1D."""
    d = dict(np.load(CLIPS, allow_pickle=True))
    d["X"] = np.clip(d["X"], -5, 5)[..., None].astype(np.float32)
    return d


if __name__ == "__main__":
    download()
    build_clips()

In [ ]:
%%writefile code/models.py
"""The two network designs. Both read one 5 s clip (625 samples) and output P(shockable)."""
import keras
from keras import layers


def small(n=625):
    """Three conv layers, 4,625 weights: designed to fit the 256 KiB / 64 KiB chip from the start."""
    return keras.Sequential([
        keras.Input((n, 1)),
        layers.Conv1D(8, 7, strides=2, activation="relu"), layers.MaxPooling1D(2),
        layers.Conv1D(16, 7, activation="relu"), layers.MaxPooling1D(2),
        layers.Conv1D(32, 7, activation="relu"), layers.GlobalAveragePooling1D(),
        layers.Dense(1, activation="sigmoid")], name="small")


def large(n=625, keep=1.0):
    """1D ResNet: a stem conv, 4 residual blocks (2 convs each) and a dense head; 501,537 weights at keep=1.

    `keep` scales the width of every layer. keep<1 gives the narrower shape that structured pruning produces,
    so the same function builds both the pruned model and its train-from-scratch twin.
    Layers are named so that pruning can copy filters across by name.
    """
    w = lambda f: max(1, round(f * keep))
    x = inp = keras.Input((n, 1))
    x = layers.Conv1D(w(32), 15, strides=2, padding="same", activation="relu", name="c0")(x)
    for i, f in enumerate([32, 64, 96, 128], 1):
        y = layers.Conv1D(w(f), 9, padding="same", activation="relu", name=f"b{i}a")(x)
        y = layers.BatchNormalization(name=f"b{i}bn")(y)
        y = layers.Conv1D(w(f), 9, padding="same", name=f"b{i}b")(y)
        # shortcut: a 1x1 conv when the channel count changes, otherwise the input itself
        x = layers.Conv1D(w(f), 1, padding="same", name=f"b{i}s")(x) if x.shape[-1] != w(f) else x
        x = layers.MaxPooling1D(2)(layers.Activation("relu")(layers.Add()([x, y])))
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(layers.Dense(w(64), activation="relu", name="d1")(x))
    out = layers.Dense(1, activation="sigmoid", name="out")(x)
    return keras.Model(inp, out, name="large" if keep == 1 else f"large_k{int(keep * 100)}")


def count_macs(model):
    """Multiply-accumulate operations for one clip (conv and dense layers; pooling/adds are negligible)."""
    total = 0
    for l in model.layers:
        if isinstance(l, layers.Conv1D):
            k, cin, cout = l.kernel.shape
            total += l.output.shape[1] * k * cin * cout
        elif isinstance(l, layers.Dense):
            total += l.kernel.shape[0] * l.kernel.shape[1]
    return int(total)

In [ ]:
%%writefile code/compress.py
"""The compression tools: structured pruning, unstructured pruning, and int8 quantisation via TensorFlow Lite."""
import gzip, time
import numpy as np, tensorflow as tf
import keras
from models import large


# ---------------------------------------------------------------- structured pruning (whole filters removed)

def l1(kernel):
    """L1 norm (sum of |weights|) of each output filter, divided by the layer mean so layers can be added."""
    s = np.abs(kernel).reshape(-1, kernel.shape[-1]).sum(0)
    return s / s.mean()


def top(score, k):
    """Indices of the k highest scores, in their original order."""
    return np.sort(np.argsort(score)[::-1][:k])


def prune_structured(model, keep):
    """Keep the highest-L1 filters in every layer and copy their weights into a narrower copy of the network.

    Removing filter j of a layer also removes input channel j of the next layer. Channels that meet at a
    residual Add must be chosen together, so both branches of an Add use one shared index set.
    """
    narrow = large(model.input.shape[1], keep)
    W = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    width = lambda name: narrow.get_layer(name).get_weights()[0].shape[-1]
    new = {}
    # Block 1 has no shortcut conv: the stem c0 is added straight to b1b, so they share one index set.
    out_idx = top(l1(W["c0"][0]) + l1(W["b1b"][0]), width("c0"))
    new["c0"] = [W["c0"][0][:, :, out_idx], W["c0"][1][out_idx]]
    in_idx = out_idx
    for i in range(1, 5):
        a = top(l1(W[f"b{i}a"][0]), width(f"b{i}a"))
        new[f"b{i}a"] = [W[f"b{i}a"][0][:, in_idx][:, :, a], W[f"b{i}a"][1][a]]
        new[f"b{i}bn"] = [p[a] for p in W[f"b{i}bn"]]
        if i > 1:  # blocks 2-4: 1x1 shortcut conv and b{i}b meet at the Add
            out_idx = top(l1(W[f"b{i}b"][0]) + l1(W[f"b{i}s"][0]), width(f"b{i}b"))
            new[f"b{i}s"] = [W[f"b{i}s"][0][:, in_idx][:, :, out_idx], W[f"b{i}s"][1][out_idx]]
        new[f"b{i}b"] = [W[f"b{i}b"][0][:, a][:, :, out_idx], W[f"b{i}b"][1][out_idx]]
        in_idx = out_idx
    d = top(l1(W["d1"][0]), width("d1"))
    new["d1"] = [W["d1"][0][in_idx][:, d], W["d1"][1][d]]
    new["out"] = [W["out"][0][d], W["out"][1]]
    for name, ws in new.items():
        narrow.get_layer(name).set_weights(ws)
    return narrow


# ---------------------------------------------------------------- unstructured pruning (single weights zeroed)

def prune_unstructured(model, sparsity):
    """Zero the smallest-magnitude `sparsity` share of the kernel weights in every conv / dense layer.

    Returns a copy of the model and the masks (1 = keep, 0 = pruned). The shape of the network is unchanged.
    """
    pruned = keras.models.clone_model(model)
    pruned.set_weights(model.get_weights())
    masks = {}
    for l in pruned.layers:
        if hasattr(l, "kernel") and l.kernel.shape[-1] > 1:  # skip the single-output layer
            k = l.kernel.numpy()
            m = (np.abs(k) > np.quantile(np.abs(k), sparsity)).astype(np.float32)
            l.kernel.assign(k * m)
            masks[l.name] = m
    return pruned, masks


class KeepPruned(keras.callbacks.Callback):
    """During fine-tuning, re-apply the masks after every batch so pruned weights stay at zero."""

    def __init__(self, masks):
        super().__init__()
        self.masks = {n: tf.constant(m) for n, m in masks.items()}

    def on_train_batch_end(self, batch, logs=None):
        for n, m in self.masks.items():
            k = self.model.get_layer(n).kernel
            k.assign(k * m)


def sparsity_of(model):
    ks = [l.kernel.numpy() for l in model.layers if hasattr(l, "kernel")]
    return float(sum((k == 0).sum() for k in ks) / sum(k.size for k in ks))


# ---------------------------------------------------------------- quantisation (TensorFlow Lite converter)

def to_tflite_fp32(model):
    """Plain conversion, no compression: the float32 reference file."""
    return tf.lite.TFLiteConverter.from_keras_model(model).convert()


def to_tflite_dynamic(model):
    """Dynamic-range quantisation: weights stored as int8, activations stay float32. Needs no data."""
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    return conv.convert()


def to_tflite_int8(model, calib):
    """Full-integer quantisation: int8 weights (one scale per output channel) and int8 activations.

    The converter runs the calibration clips through the model to record each activation's range, which sets
    that tensor's scale and zero point. Input and output are int8 too, so the chip never touches floating point.
    """
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([c[None]] for c in calib)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = conv.inference_output_type = tf.int8
    return conv.convert()


def run_tflite(blob, X, batch=256):
    """P(shockable) for every clip in X, computed by the TFLite interpreter exactly as stored in the file."""
    it = tf.lite.Interpreter(model_content=blob)
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    out = np.empty(len(X), np.float32)
    for s in range(0, len(X), batch):
        xb = X[s:s + batch]
        if i["dtype"] == np.int8:  # float mV -> int8 using the input tensor's scale and zero point
            sc, zp = i["quantization"]
            xb = np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8)
        it.resize_tensor_input(i["index"], xb.shape)
        it.allocate_tensors()
        it.set_tensor(i["index"], xb)
        it.invoke()
        y = it.get_tensor(o["index"]).ravel()
        if o["dtype"] == np.int8:
            sc, zp = o["quantization"]
            y = (y.astype(np.float32) - zp) * sc
        out[s:s + len(xb)] = y
    return out


def latency_ms(blob, x, runs=200):
    """Median time for one clip on this computer's CPU, single thread (not the microcontroller)."""
    it = tf.lite.Interpreter(model_content=blob, num_threads=1)
    it.allocate_tensors()
    i = it.get_input_details()[0]
    q = np.zeros(i["shape"], i["dtype"]) if i["dtype"] == np.int8 else x[None].astype(np.float32)
    it.set_tensor(i["index"], q)
    for _ in range(20):
        it.invoke()
    t = []
    for _ in range(runs):
        t0 = time.perf_counter()
        it.invoke()
        t.append(time.perf_counter() - t0)
    return 1000 * float(np.median(t))


def peak_activation_bytes(blob):
    """Largest total size of the activation tensors alive at the same moment while the model runs.

    This is a lower bound on the RAM the model needs on the chip (the runtime adds some overhead on top).
    """
    it = tf.lite.Interpreter(model_content=blob)
    ops = it._get_ops_details()
    t = {d["index"]: d for d in it.get_tensor_details()}
    first = {it.get_input_details()[0]["index"]: -1}
    for k, op in enumerate(ops):
        for i in op["outputs"]:
            first[i] = k
    last = {i: (len(ops) if i == it.get_output_details()[0]["index"] else s) for i, s in first.items()}
    for k, op in enumerate(ops):
        for i in op["inputs"]:
            if i in last:
                last[i] = max(last[i], k)
    size = lambda i: int(np.prod(t[i]["shape"])) * np.dtype(t[i]["dtype"]).itemsize
    return max(sum(size(i) for i in first if first[i] <= k <= last[i]) for k in range(len(ops)))


def gzip_bytes(blob):
    """File size after gzip: zeros left by unstructured pruning compress well, but only on disk."""
    return len(gzip.compress(blob, 9))

In [ ]:
%%writefile code/evaluate.py
"""Scoring against the AHA targets, and choosing a decision threshold."""
import numpy as np, pandas as pd
from config import AHA, REPORT_ONLY, SEED

GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def lower_bound(hit, patient, rng, n=1000):
    """One-sided 90% lower confidence limit, by resampling whole patients (clips from one person are alike)."""
    pats = np.unique(patient)
    by = {p: hit[patient == p] for p in pats}
    stats = [np.concatenate([by[p] for p in rng.choice(pats, len(pats))]).mean() for _ in range(n)]
    return float(np.percentile(stats, 10))


def score(category, shocked, cat, metric):
    m = category == cat
    return shocked[m].mean() if metric == "sensitivity" else (~shocked[m]).mean()


def aha_table(category, patient, shocked):
    """One row per AHA category: observed score, its lower limit, and pass/fail.

    A row passes when the observed score beats the AHA goal AND its lower limit is at least the lowest limit
    the AHA accepts. Fine VF and other VT are reported without a goal, as the AHA asks.
    """
    rng = np.random.default_rng(SEED)
    rows = []
    for cat, metric, goal, min_lcl in AHA + [(c, "sensitivity", None, None) for c in REPORT_ONLY]:
        m = category == cat
        if not m.any():
            continue
        hit = shocked[m] if metric == "sensitivity" else ~shocked[m]
        s = float(hit.mean())
        lb = lower_bound(hit, patient[m], rng) if goal is not None else None
        rows.append(dict(category=cat, clips=int(m.sum()), patients=len(np.unique(patient[m])), metric=metric,
                         goal=goal, min_lower90=min_lcl, score=round(s, 4),
                         lower90=None if lb is None else round(lb, 4),
                         passes=None if goal is None else bool(s > goal and lb >= min_lcl)))
    return pd.DataFrame(rows)


def choose_threshold(category, prob, min_clips=20):
    """Threshold with the best worst-case margin over the AHA goals, judged on validation patients only.

    For each candidate t, margin = score - goal for every AHA category with at least `min_clips` clips;
    keep the t whose smallest margin is largest. Ties go to the t closest to 0.5.
    """
    rows = [(c, mt, g) for c, mt, g, _ in AHA if (category == c).sum() >= min_clips]
    worst = np.array([min(score(category, prob >= t, c, mt) - g for c, mt, g in rows) for t in GRID])
    best = GRID[worst >= worst.max() - 1e-9]
    return float(best[np.argmin(np.abs(best - 0.5))])

In [ ]:
%%writefile code/run_experiments.py
"""Step 2: train, compress and score every model, with patient-wise 5-fold cross-validation.

    python code/run_experiments.py --all         all 5 folds as parallel processes (about 30-40 min, 16+ cores)
    python code/run_experiments.py --fold 0      one fold only
    python code/run_experiments.py --quick       fold 0 with 2 epochs, to check the pipeline runs (results_quick/)

Per fold:
  1. train the small model and the large model (float32)
  2. structured pruning of the large model (keep 75/50/35% of filters), then fine-tune
  3. the same narrow shapes trained from scratch (does pruning beat simply starting small?)
  4. unstructured pruning of the large model (75% of weights set to zero), then fine-tune
  5. convert every model to .tflite: float32, dynamic-range int8 (small/large only) and full-integer int8
  6. score every .tflite file on the validation patients (to pick a threshold later) and the test patients
Writes results/raw/fold{k}.npz (predictions) and, for fold 0, the .tflite files and their sizes.
"""
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import argparse, json, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
import keras
from sklearn.model_selection import StratifiedGroupKFold
from config import (RESULTS, SEED, FOLDS, EPOCHS, FT_EPOCHS, BATCH, N_CALIB, KEEPS, SCRATCH_KEEPS, SPARSITY)
from data import load_clips
from models import small, large, count_macs
from compress import (prune_structured, prune_unstructured, KeepPruned, sparsity_of, to_tflite_fp32,
                         to_tflite_dynamic, to_tflite_int8, run_tflite, peak_activation_bytes, gzip_bytes)


def split(d, k):
    """Fold k: test patients, plus training patients split again into train / validation (1 in 6 patients).

    Whole patients go to one side only, so the model is always tested on people it has never seen.
    """
    y = d["shock"] == 1
    has_shock = pd.Series(y).groupby(d["patient"]).transform("max").values  # balance shockable patients
    folds = StratifiedGroupKFold(FOLDS, shuffle=True, random_state=SEED).split(d["X"], has_shock, d["patient"])
    tr, te = list(folds)[k]
    tr = tr[d["shock"][tr] >= 0]  # intermediate rhythms (fine VF, slow VT) are scored but never trained on
    pats = np.unique(d["patient"][tr])
    val_p = np.random.default_rng(SEED + k).choice(pats, max(1, len(pats) // 6), replace=False)
    in_val = np.isin(d["patient"][tr], val_p)
    return tr[~in_val], tr[in_val], te


def fit(model, X, y, tr, va, epochs, lr, callbacks=()):
    """Adam + binary cross-entropy, classes weighted equally, early stopping on validation loss."""
    w = {0: len(tr) / (2 * (y[tr] == 0).sum()), 1: len(tr) / (2 * (y[tr] == 1).sum())}
    model.compile(keras.optimizers.Adam(lr), "binary_crossentropy")
    stop = keras.callbacks.EarlyStopping("val_loss", patience=6 if lr >= 1e-3 else 4, restore_best_weights=True)
    model.fit(X[tr], y[tr], validation_data=(X[va], y[va]), epochs=epochs, batch_size=BATCH, class_weight=w,
              verbose=0, callbacks=[stop, *callbacks])
    return model


def run_fold(k, quick, out):
    t0 = time.time()
    d = load_clips()
    X, y = d["X"], (d["shock"] == 1).astype(np.float32)
    tr, va, te = split(d, k)
    epochs, ft_epochs = (2, 1) if quick else (EPOCHS, FT_EPOCHS)
    keras.utils.set_random_seed(SEED + k)
    calib = X[np.random.default_rng(SEED + k).choice(tr, N_CALIB, replace=False)]
    log = lambda msg: print(f"[fold {k} {(time.time() - t0) / 60:5.1f} min] {msg}", flush=True)
    log(f"train {len(tr)}  val {len(va)}  test {len(te)} clips")

    probs = {}  # variant name -> (validation probabilities, test probabilities)
    keras_prob = lambda m: tuple(m.predict(X[idx], batch_size=512, verbose=0).ravel() for idx in (va, te))
    nets = {"small": fit(small(), X, y, tr, va, epochs, 1e-3)}
    log("small trained")
    nets["large"] = fit(large(), X, y, tr, va, epochs, 1e-3)
    log("large trained")
    for keep in KEEPS:
        tag = f"large_p{int(keep * 100)}"
        p = prune_structured(nets["large"], keep)
        probs[f"{tag}_noft_fp32"] = keras_prob(p)  # straight after cutting, before any retraining
        nets[tag] = fit(p, X, y, tr, va, ft_epochs, 3e-4)
        log(f"{tag} pruned + fine-tuned")
    for keep in SCRATCH_KEEPS:
        nets[f"scratch_k{int(keep * 100)}"] = fit(large(keep=keep), X, y, tr, va, epochs, 1e-3)
        log(f"scratch_k{int(keep * 100)} trained")
    tag = f"large_u{int(SPARSITY * 100)}"
    u, masks = prune_unstructured(nets["large"], SPARSITY)
    probs[f"{tag}_noft_fp32"] = keras_prob(u)
    nets[tag] = fit(u, X, y, tr, va, ft_epochs, 3e-4, [KeepPruned(masks)])
    log(f"{tag} pruned + fine-tuned (sparsity {sparsity_of(nets[tag]):.3f})")

    mdir = out / "models"
    mdir.mkdir(parents=True, exist_ok=True)
    sizes = {}
    for fam, m in nets.items():
        blobs = {"fp32": to_tflite_fp32(m), "int8": to_tflite_int8(m, calib)}
        if fam in ("small", "large"):
            blobs["dyn"] = to_tflite_dynamic(m)
        for kind, blob in blobs.items():
            name = f"{fam}_{kind}"
            probs[name] = (run_tflite(blob, X[va]), run_tflite(blob, X[te]))
            if k == 0:
                (mdir / f"{name}.tflite").write_bytes(blob)
                sizes[name] = dict(params=m.count_params(), macs=count_macs(m), sparsity=round(sparsity_of(m), 4),
                                   file_kib=len(blob) / 1024, gzip_kib=gzip_bytes(blob) / 1024,
                                   peak_ram_kib=peak_activation_bytes(blob) / 1024)
        if k == 0 and fam == "large":
            m.save(mdir / "large_fp32.keras")  # kept for the weight / filter figures
    log("converted and scored all .tflite files")

    raw = out / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(raw / f"fold{k}.npz", val_idx=va, test_idx=te,
                        **{f"{n}__val": v for n, (v, _) in probs.items()},
                        **{f"{n}__test": t for n, (_, t) in probs.items()})
    if k == 0:
        json.dump(sizes, open(out / "sizes.json", "w"), indent=1)
    log("done")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fold", type=int)
    ap.add_argument("--all", action="store_true")
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--threads", type=int, default=0, help="CPU threads per process (0 = TensorFlow default)")
    a = ap.parse_args()
    out = RESULTS.parent / "results_quick" if a.quick else RESULTS
    if a.all:  # one process per fold, CPU threads shared out between them
        threads = a.threads or max(1, (os.cpu_count() or 4) // FOLDS)
        (out / "raw").mkdir(parents=True, exist_ok=True)
        procs = [subprocess.Popen([sys.executable, __file__, "--fold", str(k), "--threads", str(threads)],
                                  stdout=open(out / "raw" / f"fold{k}.log", "w"), stderr=subprocess.STDOUT)
                 for k in range(FOLDS)]
        codes = [p.wait() for p in procs]
        print("exit codes per fold:", codes)
        sys.exit(max(codes))
    if a.threads:
        tf.config.threading.set_intra_op_parallelism_threads(a.threads)
        tf.config.threading.set_inter_op_parallelism_threads(2)
    run_fold(0 if a.fold is None else a.fold, a.quick, out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/summarise.py
"""Step 3: turn the per-fold predictions into result tables.

    python code/summarise.py

Every test clip was predicted by a model that never saw that patient, so all 5 folds are pooled and scored
together. Each model is scored twice: with the decision threshold fixed at 0.5, and with a threshold chosen
per fold on that fold's validation patients (never on the test patients).
Writes results/tables/*.csv, results/thresholds.json and results/RESULTS.md.
"""
import json, re, sys
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
import config
from config import CLIPS, AHA, FLASH_BYTES, RAM_BYTES, MCU_HZ, MACS_PER_CYCLE
from compress import latency_ms
from evaluate import aha_table, choose_threshold

STAGES = ["fp32", "dyn", "noft_fp32", "int8"]
FAMILIES = ["small", "large", "large_p75", "large_p50", "large_p35", "scratch_k50", "scratch_k35", "large_u75"]
SHOWN = ["coarse VF", "rapid VT", "NSR", "other non-shockable", "asystole", "fine VF", "other VT"]


def split_name(name):
    fam = re.match(r"(small|large_p\d+|large_u\d+|large|scratch_k\d+)_(.*)", name)
    return fam.group(1), fam.group(2)


def md_table(df):
    """DataFrame -> Markdown table (no extra packages)."""
    fmt = lambda v: "" if v is None or (isinstance(v, float) and np.isnan(v)) else (f"{v:.3f}" if isinstance(v, float) else str(v))
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


RESULTS = config.RESULTS.parent / "results_quick" if "--quick" in sys.argv else config.RESULTS


def main():
    z = np.load(CLIPS, allow_pickle=True)
    cat, pat, shock = z["category"], z["patient"], z["shock"]
    folds = sorted((RESULTS / "raw").glob("fold*.npz"))
    sizes = json.load(open(RESULTS / "sizes.json"))
    raw = [np.load(f) for f in folds]
    names = sorted({k.split("__")[0] for k in raw[0].files if "__" in k},
                   key=lambda n: (FAMILIES.index(split_name(n)[0]), STAGES.index(split_name(n)[1])))
    print(f"{len(folds)} folds, {len(names)} model variants")

    prob, thr_at, thresholds = {}, {}, {}
    for name in names:
        p, t = np.full(len(cat), np.nan, np.float32), np.full(len(cat), np.nan, np.float32)
        thresholds[name] = []
        for r in raw:
            va, te = r["val_idx"], r["test_idx"]
            th = choose_threshold(cat[va], r[f"{name}__val"])
            thresholds[name].append(th)
            p[te], t[te] = r[f"{name}__test"], th
        prob[name], thr_at[name] = p, t
    done = ~np.isnan(prob[names[0]])

    tables, rows = [], []
    for name in names:
        fam, stage = split_name(name)
        row = dict(model=name, family=fam, stage=stage)
        s = sizes.get(name, {})
        row.update(params=s.get("params"), macs=s.get("macs"), sparsity=s.get("sparsity"), file_kib=s.get("file_kib"),
                   gzip_kib=s.get("gzip_kib"), peak_ram_kib=s.get("peak_ram_kib"))
        f = RESULTS / "models" / f"{name}.tflite"
        row["pc_ms"] = latency_ms(f.read_bytes(), np.zeros((625, 1), np.float32)) if f.exists() else None
        if stage == "int8":
            row["mcu_ms_low"] = 1000 * s["macs"] / (MACS_PER_CYCLE[1] * MCU_HZ)
            row["mcu_ms_high"] = 1000 * s["macs"] / (MACS_PER_CYCLE[0] * MCU_HZ)
            row["fits_chip"] = bool(s["file_kib"] * 1024 <= FLASH_BYTES and s["peak_ram_kib"] * 1024 <= RAM_BYTES)
        both = done & (shock >= 0)
        row["auroc"] = roc_auc_score(shock[both], prob[name][both])
        for mode, shocked in (("0.5", prob[name] >= 0.5), ("tuned", prob[name] >= thr_at[name])):
            tab = aha_table(cat[done], pat[done], shocked[done])
            tab.insert(0, "threshold", mode)
            tab.insert(0, "model", name)
            tables.append(tab)
            graded = tab[tab["passes"].notna()]
            row[f"sens_all_{mode}"] = float(shocked[done & (shock == 1)].mean())   # all shockable clips
            row[f"spec_all_{mode}"] = float((~shocked[done & (shock == 0)]).mean())  # all non-shockable clips
            row[f"passed_{mode}"] = int(graded["passes"].sum())
            row[f"worst_margin_{mode}"] = float((graded["score"] - graded["goal"]).min())
            if mode == "tuned":
                row.update({c: v for c, v in zip(tab["category"], tab["score"])})
        row["threshold_mean"] = float(np.mean(thresholds[name]))
        own = prob.get(f"{fam}_fp32")
        row["flips_vs_own_fp32"] = int(((prob[name] >= 0.5) != (own >= 0.5))[done].sum()) if own is not None else None
        row["flips_vs_large_fp32"] = int(((prob[name] >= 0.5) != (prob["large_fp32"] >= 0.5))[done].sum())
        rows.append(row)

    out = RESULTS / "tables"
    out.mkdir(exist_ok=True)
    summary = pd.DataFrame(rows)
    summary.to_csv(out / "summary.csv", index=False)
    pd.concat(tables).to_csv(out / "aha_by_model.csv", index=False)
    json.dump(thresholds, open(RESULTS / "thresholds.json", "w"), indent=1)
    meta = pd.DataFrame(dict(category=cat, patient=pat, db=z["db"]))
    data = meta.pivot_table(index="category", columns="db", values="patient", aggfunc="count", fill_value=0)
    data["clips"] = data.sum(axis=1)
    data["patients"] = meta.groupby("category")["patient"].nunique()
    data = data.reindex(SHOWN).reset_index()
    data.to_csv(out / "dataset.csv", index=False)

    # ---- RESULTS.md: the human-readable version
    goals = {c: f"{'Se' if m == 'sensitivity' else 'Sp'} > {g:.0%}" for c, m, g, _ in AHA}
    main_cols = ["model", "params", "file_kib", "peak_ram_kib", "fits_chip", "auroc", "sens_all_0.5", "spec_all_0.5",
                 "sens_all_tuned", "spec_all_tuned", "passed_0.5", "passed_tuned",
                 "worst_margin_tuned"] + [c for c in SHOWN if c in summary]
    md = ["# Results", "", "Generated by `code/summarise.py` from `results/raw/`. Pooled out-of-fold predictions from "
          f"patient-wise {len(folds)}-fold cross-validation.", "",
          "## Dataset", "", md_table(data), "",
          "## All models", "",
          "`passed_*` = AHA rows passed out of 5 (score beats the goal and the lower 90% bound reaches the AHA minimum). "
          "Per-category columns are scores at the tuned threshold. AHA goals: "
          + "; ".join(f"{c} {g}" for c, g in goals.items()) + ".", "",
          md_table(summary[main_cols]), "",
          "## Size, memory and speed (int8 files)", "",
          f"Chip budget: {FLASH_BYTES // 1024} KiB flash, {RAM_BYTES // 1024} KiB RAM, {MCU_HZ / 1e6:.0f} MHz. "
          "`mcu_ms` is an estimate from the MAC count at 0.5-1 MAC per cycle; `pc_ms` is measured on a laptop CPU.", "",
          md_table(summary[summary.stage == "int8"][["model", "params", "macs", "file_kib", "gzip_kib", "peak_ram_kib",
                                                    "fits_chip", "mcu_ms_low", "mcu_ms_high", "pc_ms"]]), "",
          "## Decision thresholds chosen on validation patients (per fold)", "",
          md_table(pd.DataFrame([dict(model=n, **{f"fold{i}": t for i, t in enumerate(v)}) for n, v in thresholds.items()]))]
    (RESULTS / "RESULTS.md").write_text("\n".join(md), encoding="utf-8")
    with pd.option_context("display.width", 250, "display.max_columns", 40):
        print(summary[main_cols].to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/make_figures.py
"""Step 4: draw the report figures from the results.

    python code/make_figures.py          (after summarise.py)
Writes results/figures/fig*.png.
"""
import os, sys
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import config
from config import CLIPS, AHA, FS, FLASH_BYTES
from compress import l1

RESULTS = config.RESULTS.parent / "results_quick" if "--quick" in sys.argv else config.RESULTS
FIG = RESULTS / "figures"
# Validated categorical slots (blue, orange, aqua, yellow) + ink colours; light theme for print
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "savefig.dpi": 200, "savefig.bbox": "tight", "axes.titlesize": 9.5, "axes.titleweight": "bold"})
NICE = {"small": "Small (trained small)", "large": "Large", "large_p75": "Large, pruned to 75%",
        "large_p50": "Large, pruned to 50%", "large_p35": "Large, pruned to 35%",
        "scratch_k50": "50%-width, from scratch", "scratch_k35": "35%-width, from scratch",
        "large_u75": "Large, 75% weights zeroed"}
SHORT = {"small": "Small", "large": "Large", "large_p75": "Pruned 75%", "large_p50": "Pruned 50%",
         "large_p35": "Pruned 35%", "scratch_k50": "Scratch 50%", "scratch_k35": "Scratch 35%",
         "large_u75": "Zeroed 75%"}


def save(fig, name):
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / name)
    plt.close(fig)
    print("wrote", FIG / name)


def fig_examples(z):
    """One 5 s clip of each AHA category, same scale, so amplitude and rate differences are visible."""
    cats = ["coarse VF", "rapid VT", "NSR", "other non-shockable", "asystole"]
    fig, axes = plt.subplots(len(cats), 1, figsize=(6.5, 6), sharex=True, sharey=True)
    t = np.arange(z["X"].shape[1]) / FS
    for ax, c in zip(axes, cats):
        idx = np.flatnonzero(z["category"] == c)
        amp = np.ptp(z["X"][idx], axis=1)
        i = idx[np.argsort(amp)[len(idx) // 2]]  # a typical clip: median peak-to-peak amplitude
        ax.plot(t, z["X"][i], color=C1, lw=1)
        shock = "shock" if c in ("coarse VF", "rapid VT") else "no shock"
        ax.set_title(f"{c}  ({shock})  -  {z['patient'][i]}", loc="left", fontweight="normal", color=INK)
        ax.set_ylim(-3, 3)
        ax.set_ylabel("mV")
    axes[-1].set_xlabel("time (s)")
    fig.tight_layout()
    save(fig, "fig1_example_clips.png")


def fig_quantisation(raw, summary):
    """(a) one layer's weights and the int8 grid; (b) float32 vs int8 output for the same test clips."""
    import keras
    m = keras.models.load_model(RESULTS / "models" / "large_fp32.keras")
    w = m.get_layer("b2a").kernel.numpy()[..., 0].ravel()  # all weights of output filter 0 in layer b2a
    step = np.abs(w).max() / 127  # symmetric per-channel int8: one step = max|w| / 127
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 3))
    a.hist(w, bins=60, color=C1, edgecolor="white", linewidth=0.5)
    a.set_title("(a) 288 weights of one filter", loc="left")
    a.set_xlabel("weight value")
    a.set_ylabel("count")
    a.text(0.02, 0.97, f"int8 step = max|w| / 127 = {step:.5f}", transform=a.transAxes, va="top", color=INK,
           fontsize=8, bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
    a.set_ylim(0, a.get_ylim()[1] * 1.2)
    te = np.concatenate([r["test_idx"] for r in raw])
    p32 = np.concatenate([r["large_fp32__test"] for r in raw])
    p8 = np.concatenate([r["large_int8__test"] for r in raw])
    b.scatter(p32, p8, s=8, color=C1, alpha=0.35, edgecolors="none")
    b.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--")
    flips = int(((p32 >= 0.5) != (p8 >= 0.5)).sum())
    b.set_title("(b) Large model: float32 vs int8 output", loc="left")
    b.set_xlabel("P(shock), float32")
    b.set_ylabel("P(shock), int8")
    b.text(0.03, 0.95, f"{len(te):,} test clips\n{flips} decisions flip at 0.5", transform=b.transAxes, va="top",
           color=INK2, fontsize=8)
    fig.tight_layout()
    save(fig, "fig3_quantisation.png")


def fig_pruning_l1():
    """Sorted filter L1 norms in one layer, with the filters a 50% prune keeps."""
    import keras
    m = keras.models.load_model(RESULTS / "models" / "large_fp32.keras")
    s = np.sort(l1(m.get_layer("b4a").kernel.numpy()))[::-1]
    keep = len(s) // 2
    fig, ax = plt.subplots(figsize=(6.5, 2.6))
    ax.bar(np.arange(keep), s[:keep], color=C1, width=0.85, label=f"kept ({keep})")
    ax.bar(np.arange(keep, len(s)), s[keep:], color="#c3c2b7", width=0.85, label=f"removed ({len(s) - keep})")
    ax.set_xlabel("filter, sorted by L1 norm")
    ax.set_ylabel("L1 norm / layer mean")
    ax.set_title("Structured pruning at 50%: layer b4a keeps its 64 strongest of 128 filters", loc="left")
    ax.legend(frameon=False)
    fig.tight_layout()
    save(fig, "fig4_pruning_l1.png")


def fig_pruning_recovery(summary):
    """AUROC straight after cutting vs after fine-tuning, per pruning level."""
    s = summary.set_index("model")["auroc"]
    keeps = [100, 75, 50, 35]
    noft = [s["large_fp32"]] + [s[f"large_p{k}_noft_fp32"] for k in keeps[1:]]
    ft = [s["large_fp32"]] + [s[f"large_p{k}_fp32"] for k in keeps[1:]]
    fig, ax = plt.subplots(figsize=(4.6, 3))
    ax.plot(keeps, noft, color=C2, lw=2, marker="o", ms=6, label="cut, no retraining")
    ax.plot(keeps, ft, color=C1, lw=2, marker="o", ms=6, label="cut + 20 epochs fine-tuning")
    for x, y in zip(keeps[1:], noft[1:]):
        ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(6, -10), color=INK2, fontsize=8)
    ax.invert_xaxis()
    ax.set_xticks(keeps)
    ax.set_xlabel("% of filters kept in every layer")
    ax.set_ylabel("AUROC (shock vs no shock)")
    ax.set_ylim(min(0.4, min(noft) - 0.05), 1.02)
    ax.set_title("Pruning damages the model; fine-tuning repairs it", loc="left")
    ax.legend(frameon=False, loc="lower left")
    fig.tight_layout()
    save(fig, "fig5_pruning_recovery.png")


def fig_headline(summary):
    """AHA rows passed vs int8 file size: compressed-large models vs models trained small."""
    s = summary[summary.stage == "int8"].copy()
    fig, ax = plt.subplots(figsize=(6.8, 3.6))
    groups = [("compressed from the large model", ["large", "large_p75", "large_p50", "large_p35", "large_u75"], C1, "o"),
              ("trained small from the start", ["scratch_k50", "scratch_k35", "small"], C2, "s")]
    for label, fams, col, mk in groups:
        g = s[s.family.isin(fams)].copy()
        # pruned and scratch twins (and large / zeroed) share a file size: nudge them apart so both show
        g["file_kib"] *= g.family.map({"large_p50": 0.94, "large_p35": 0.94, "scratch_k50": 1.06, "scratch_k35": 1.06,
                                       "large": 0.94, "large_u75": 1.06}).fillna(1.0)
        ax.scatter(g.file_kib, g["passed_0.5"], s=60, color=col, marker=mk, label=label, zorder=3,
                   edgecolors="white", linewidths=1.5)
        for _, r in g.iterrows():  # short labels; points sharing a size are split above / below
            dy = {"large_u75": -13, "scratch_k35": -13, "scratch_k50": -13, "large_p75": -13}.get(r.family, 9)
            ax.annotate(SHORT[r.family], (r.file_kib, r["passed_0.5"]), textcoords="offset points",
                        xytext=(-4 if dy > 0 else 4, dy), fontsize=7.5, color=INK2, va="center",
                        ha="right" if dy > 0 else "left")
    ax.axvline(FLASH_BYTES / 1024, color=INK, lw=1, ls="--")
    ax.text(FLASH_BYTES / 1024 * 0.93, 5.25, "chip flash: 256 KiB", ha="right", color=INK, fontsize=8)
    ax.set_xscale("log")
    ticks = [10, 25, 50, 100, 250, 500, 1000]
    ax.set_xticks(ticks, [str(t) for t in ticks])
    ax.minorticks_off()
    ax.set_xlim(8, 1500)
    ax.set_xlabel("int8 model file (KiB, log scale)")
    ax.set_ylabel("AHA targets passed (of 5)")
    ax.set_ylim(-0.3, 5.5)
    ax.set_yticks(range(6))
    ax.set_title("Which int8 models fit the chip, and how many AHA targets they meet", loc="left")
    ax.legend(frameon=False, loc="lower left", fontsize=8)
    fig.tight_layout()
    save(fig, "fig7_headline.png")


def fig_per_category(aha):
    """Score per AHA category for the main contenders, against each goal."""
    models = [("large_fp32", "Large, float32 (reference, too big)", C1), ("large_p35_int8", "Large pruned to 35% + int8", C3),
              ("scratch_k35_int8", "35%-width from scratch + int8", C2), ("small_int8", "Small + int8", C4)]
    cats = [c for c, *_ in AHA]
    t = aha[aha.threshold.astype(str) == "0.5"]
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    w = 0.19
    for j, (m, label, col) in enumerate(models):
        r = t[t.model == m].set_index("category").reindex(cats)
        ax.bar(np.arange(len(cats)) + (j - 1.5) * w, r["score"], width=w - 0.02, color=col, label=label)
    for i, (c, metric, goal, _) in enumerate(AHA):
        ax.plot([i - 2.1 * w, i + 2.1 * w], [goal, goal], color=INK, lw=1.5)
    ax.plot([], [], color=INK, lw=1.5, label="AHA goal")
    ax.set_xticks(range(len(cats)))
    ax.set_xticklabels([f"{c}\n({'Se' if m == 'sensitivity' else 'Sp'})" for c, m, *_ in AHA], fontsize=8)
    low = t[t.model.isin([m for m, *_ in models]) & t.category.isin(cats)]["score"].min()
    ax.set_ylim(max(0.0, min(0.5, np.floor((low - 0.05) * 10) / 10)), 1.02)
    ax.set_ylabel("score (threshold 0.5)")
    ax.set_title("Score per AHA rhythm category", loc="left", pad=30)
    ax.legend(frameon=False, fontsize=7.5, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0))
    fig.tight_layout()
    save(fig, "fig8_per_category.png")


def fig_structured_vs_unstructured(summary):
    """Same 75% of weights removed: zeros (unstructured) vs a smaller network (structured)."""
    s = summary.set_index("model")
    rows = [("large_int8", "Large\n(no pruning)"), ("large_u75_int8", "75% of weights\nzeroed (unstructured)"),
            ("large_p50_int8", "50% of filters removed\n(structured, ~75% fewer weights)")]
    x = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(6.2, 3))
    ax.bar(x - 0.2, [s.loc[m, "file_kib"] for m, _ in rows], 0.38, color=C1, label="int8 file (what the chip stores)")
    ax.bar(x + 0.2, [s.loc[m, "gzip_kib"] for m, _ in rows], 0.38, color=C3, label="same file gzipped (disk only)")
    for i, (m, _) in enumerate(rows):
        ax.text(i - 0.2, s.loc[m, "file_kib"] + 8, f"{s.loc[m, 'file_kib']:.0f}", ha="center", fontsize=8, color=INK)
        ax.text(i + 0.2, s.loc[m, "gzip_kib"] + 8, f"{s.loc[m, 'gzip_kib']:.0f}", ha="center", fontsize=8, color=INK)
    ax.axhline(FLASH_BYTES / 1024, color=INK, lw=1, ls="--")
    ax.text(2.45, FLASH_BYTES / 1024 + 8, "256 KiB flash", ha="right", fontsize=8, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels([l for _, l in rows], fontsize=8)
    ax.set_ylabel("KiB")
    ax.set_title("Zeros only shrink a compressed file; removing filters shrinks the model", loc="left")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save(fig, "fig6_structured_vs_unstructured.png")


def fig_pipeline():
    """Boxes-and-arrows overview of the pipeline, with the tool used at each step."""
    from matplotlib.patches import FancyBboxPatch
    steps = [("1. Data", "PhysioNet CUDB, VFDB,\nMIT-BIH → 5 s clips,\n125 Hz, AHA labels", "wfdb, SciPy"),
             ("2. Train", "large (501k weights)\nsmall (4.6k weights)\npatient-wise 5 folds", "TensorFlow / Keras"),
             ("3. Prune", "remove lowest-L1 filters\n(keep 75/50/35%)\nthen fine-tune", "NumPy + Keras"),
             ("4. Quantise", "float32 → int8\ncalibrate on 300\ntraining clips", "TFLite converter"),
             ("5. Score", "run the .tflite file\nper AHA category,\nsize, RAM, speed", "TFLite interpreter")]
    fig, ax = plt.subplots(figsize=(9.5, 2.4))
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 1)
    ax.axis("off")
    for i, (title, body, tool) in enumerate(steps):
        x = i + 0.06
        ax.add_patch(FancyBboxPatch((x, 0.12), 0.8, 0.8, boxstyle="round,pad=0.0,rounding_size=0.06",
                                    facecolor="#eef4fc", edgecolor=C1, linewidth=1.2))
        ax.text(x + 0.4, 0.83, title, ha="center", va="top", fontsize=8.5, fontweight="bold", color=INK)
        ax.text(x + 0.4, 0.66, body, ha="center", va="top", fontsize=7, color=INK, linespacing=1.3)
        ax.text(x + 0.4, 0.2, tool, ha="center", va="center", fontsize=6.5, color=C1, style="italic")
        if i < len(steps) - 1:
            ax.annotate("", xy=(i + 1.05, 0.52), xytext=(x + 0.81, 0.52),
                        arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.2))
    save(fig, "fig2_pipeline.png")


def fig_asystole_input(z):
    """An asystole clip before and after the int8 input rounding (step = input scale of the .tflite file)."""
    import tensorflow as tf
    it = tf.lite.Interpreter(model_path=str(RESULTS / "models" / "large_p35_int8.tflite"))
    sc, zp = it.get_input_details()[0]["quantization"]
    idx = np.flatnonzero(z["category"] == "asystole")
    x = z["X"][idx[np.argsort(np.ptp(z["X"][idx], axis=1))[len(idx) // 2]]]
    q = (np.clip(np.round(x / sc + zp), -128, 127) - zp) * sc
    t = np.arange(len(x)) / FS
    fig, ax = plt.subplots(figsize=(6.8, 2.6))
    ax.plot(t, 1000 * x, color=MUTED, lw=1, label="float32 input")
    ax.step(t, 1000 * q, color=C2, lw=1.4, where="mid", label=f"int8 input (step = {1000 * sc:.0f} µV)")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("µV")
    ax.set_title("Asystole after int8 rounding: a flat line becomes a few integer levels", loc="left")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout()
    save(fig, "fig9_asystole_input.png")


def main():
    fig_pipeline()
    z = np.load(CLIPS, allow_pickle=True)
    summary = pd.read_csv(RESULTS / "tables" / "summary.csv")
    aha = pd.read_csv(RESULTS / "tables" / "aha_by_model.csv")
    raw = [np.load(f) for f in sorted((RESULTS / "raw").glob("fold*.npz"))]
    fig_examples(z)
    fig_quantisation(raw, summary)
    fig_pruning_l1()
    fig_pruning_recovery(summary)
    fig_headline(summary)
    fig_per_category(aha)
    fig_structured_vs_unstructured(summary)
    fig_asystole_input(z)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/demo.py
"""Demo: give the int8 model a 5 s ECG and get the AED's answer, "SHOCK ADVISED" or "NO SHOCK".

    python code/demo.py                                  one clip of every rhythm, from patients the model never saw
    python code/demo.py --record cudb/cu12 --start 260   any PhysioNet record, from a chosen second
    python code/demo.py --model results/models/small_int8.tflite

The model is the fold-0 .tflite file, the same bytes that would be flashed to the microcontroller, and the
threshold is 0.5, as in the report's headline results (change it with --threshold).
"""
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import argparse, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, wfdb
from config import RESULTS, RAW, WIN, FS
from compress import run_tflite
from data import load_clips, preprocess, LEAD

DEFAULT = RESULTS / "models" / "large_p35_int8.tflite"  # best int8 model that fits the chip


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=Path, default=DEFAULT)
    ap.add_argument("--record", help="database/record, e.g. cudb/cu12")
    ap.add_argument("--start", type=float, default=0.0, help="start time in seconds")
    ap.add_argument("--threshold", type=float, default=0.5)
    a = ap.parse_args()
    blob = a.model.read_bytes()
    name = a.model.stem
    thr = a.threshold
    print(f"model {name}: {len(blob) / 1024:.0f} KiB, shock if P(shock) >= {thr:.2f}\n")

    if a.record:
        db, rec = a.record.split("/")
        sig, f = wfdb.rdsamp(str(RAW / db / rec))
        x = preprocess(sig[:, f["sig_name"].index(LEAD[db])], f["fs"])
        s = int(a.start * FS)
        clip = np.clip(x[s:s + WIN], -5, 5).astype(np.float32)[None, :, None]
        t0 = time.perf_counter()
        p = run_tflite(blob, clip)[0]
        print(f"{a.record} from {a.start:.0f} s: P(shock) = {p:.3f} -> {'SHOCK ADVISED' if p >= thr else 'NO SHOCK'}"
              f"   ({1000 * (time.perf_counter() - t0):.0f} ms on this PC)")
        return

    d = load_clips()
    raw = a.model.resolve().parent.parent / "raw" / "fold0.npz"  # the results folder this model came from
    test = np.load(raw)["test_idx"]  # patients held out of fold 0's training
    rng = np.random.default_rng()
    print(f"{'rhythm':<22}{'patient':<13}{'should':<10}{'P(shock)':>9}  answer")
    for cat in ["coarse VF", "rapid VT", "NSR", "other non-shockable", "asystole", "fine VF", "other VT"]:
        idx = test[d["category"][test] == cat]
        if not len(idx):
            continue
        i = rng.choice(idx)
        p = run_tflite(blob, d["X"][i:i + 1])[0]
        should = {1: "shock", 0: "no shock", -1: "(either)"}[int(d["shock"][i])]
        answer = "SHOCK ADVISED" if p >= thr else "NO SHOCK"
        ok = "" if d["shock"][i] < 0 else ("  ok" if (p >= thr) == (d["shock"][i] == 1) else "  WRONG")
        print(f"{cat:<22}{d['patient'][i]:<13}{should:<10}{p:>9.3f}  {answer}{ok}")


if __name__ == "__main__":
    main()

## 2. Download the data and cut it into clips

About 130 MB from PhysioNet (CUDB, VFDB and MIT-BIH Arrhythmia). The result is `data/clips.npz`: 23,092 five-second clips from 100 patients.

In [ ]:
!python code/data.py

## 3. Train, compress and score

Choose how much to run:

- **`quick`** (default): fold 0 only, 2 training epochs. Checks that every step works (roughly 15-30 min on Colab). The numbers will be poor because the models barely train.
- **`fold0`**: fold 0 with the full training budget (40 epochs, 20 for fine-tuning). Gives real models for the demo (roughly 1-2 h with a GPU).
- **`full`**: all 5 folds, as in the report (several hours on Colab; the report's numbers came from a 16-core PC).

Every patient is in only one of training, validation or test, so the model is always tested on people it has never seen.

In [ ]:
MODE = "quick"  #@param ["quick", "fold0", "full"]
import subprocess, sys
if MODE == "quick":
    !python code/run_experiments.py --quick
elif MODE == "fold0":
    !python code/run_experiments.py --fold 0
else:
    for k in range(5):
        !python code/run_experiments.py --fold {k}
OUT = "results_quick" if MODE == "quick" else "results"
FLAG = "--quick" if MODE == "quick" else ""

## 4. Result tables

`summary.csv` has one row per model: size, RAM, whether it fits the chip, and how many AHA rows it passes. With `quick` or `fold0` the confidence limits come from one fold's test patients only, so they are wider than in the report.

In [ ]:
!python code/summarise.py {FLAG}
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/RESULTS.md", encoding="utf-8").read()))

## 5. Figures

In [ ]:
!python code/make_figures.py {FLAG}
from IPython.display import Image
import glob
for f in sorted(glob.glob(f"{OUT}/figures/*.png")):
    print(f); display(Image(f, width=800))

## 6. Demo: the AED's answer

The int8 model with 35% of the large model's filters (93 KiB) looks at one clip of each rhythm from test patients it never saw, then at a stretch of ventricular fibrillation from record `cudb/cu12`.

In [ ]:
MODEL = f"{OUT}/models/large_p35_int8.tflite"
!python code/demo.py --model {MODEL}
!python code/demo.py --model {MODEL} --record cudb/cu12 --start 260

## 7. Download the results (optional)

Zips the tables, figures and `.tflite` models so you can keep them.

In [ ]:
!zip -qr results.zip {OUT}
from google.colab import files
files.download("results.zip")